In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold

In [7]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.feature_selection import SequentialFeatureSelector

In [5]:
df = pd.read_csv("../../data/processed/ml_dataset.csv")

print(df.shape)
display(df.head())

print(df.columns.tolist())

target_col = "target_return_5d"

exclude_cols = [
    "stock_code",
    "trade_date",
    target_col
]

X = df.drop(columns=exclude_cols)
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

(2685, 28)


,stock_code,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,...,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,target_return_5d
0,660,2024-06-11,0.021635,0.094233,0.054591,0.181212,0.011905,0.043689,0.009615,203000.0,...,6187.902397,5195.016563,992.885834,0.031048,0.023996,6757.468051,0.253659,3435519.80,0.893653,0.103529
1,660,2024-06-12,0.011765,0.112261,0.061728,0.169750,0.014151,0.023697,-0.002353,207340.0,...,6911.664995,5538.346249,1373.318746,0.028770,0.023814,6631.934619,-0.299278,3381585.20,0.636190,0.086047
2,660,2024-06-13,0.032558,0.146102,0.096296,0.198057,-0.017699,0.034247,0.051163,213000.0,...,7958.354609,6022.347921,1936.006687,0.026692,0.024432,6979.653575,1.685444,3532295.70,1.635559,0.069820
3,660,2024-06-14,-0.004505,0.065060,0.129280,0.145078,-0.017778,0.041667,0.013514,215700.0,...,8607.944994,6539.467336,2068.477659,0.014806,0.023386,7123.964034,-0.426854,3443697.95,0.961531,0.058824
4,660,2024-06-17,0.009050,0.072115,0.178647,0.174302,0.018265,0.047945,-0.009050,218700.0,...,9178.331251,7067.240119,2111.091132,0.013915,0.022745,7365.109460,-0.336094,3411545.25,0.644382,0.000000


['stock_code', 'trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'target_return_5d']
X shape: (2685, 25)
y shape: (2685,)

Features:
['return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']


In [8]:
xgb = XGBRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

n_features_list = [5, 10, 15, 20]
tol_list = [0, 0.001, 0.005]

In [11]:
results = []

feature_counts = [5, 10, 15, 20]
tol_values = [0, 0.001, 0.005]

for n_features in feature_counts:
    for tol in tol_values:

        print(
            f"\n===== SFS + XGBoost | "
            f"n_features={n_features}, tol={tol} ====="
        )

        xgb = XGBRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )

        sfs = SequentialFeatureSelector(
            xgb,
            n_features_to_select=n_features,
            direction="forward",
            scoring="neg_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            tol=tol
        )

        sfs.fit(X, y)

        selected_features = X.columns[sfs.get_support()].tolist()

        print("Selected features:")
        print(selected_features)

        results.append({
            "n_features_to_select": n_features,
            "tol": tol,
            "n_selected_features": len(selected_features),
            "selected_features": selected_features
        })

sfs_xgb_results = pd.DataFrame(results)

display(sfs_xgb_results)


===== SFS + XGBoost | n_features=5, tol=0 =====
Selected features:
['sma_5', 'sma_60', 'price_to_sma_60', 'volatility_20', 'volume_sma_20']

===== SFS + XGBoost | n_features=5, tol=0.001 =====
Selected features:
['sma_5', 'sma_60', 'price_to_sma_60', 'volatility_20', 'volume_sma_20']

===== SFS + XGBoost | n_features=5, tol=0.005 =====
Selected features:
['sma_5', 'sma_60', 'price_to_sma_60', 'volatility_20', 'volume_sma_20']

===== SFS + XGBoost | n_features=10, tol=0 =====
Selected features:
['sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']

===== SFS + XGBoost | n_features=10, tol=0.001 =====
Selected features:
['sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']

===== SFS + XGBoost | n_features=10, tol=0.005 =====
Selected features:
['sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility

,n_features_to_select,tol,n_selected_features,selected_features
0,5,0.000,5,"[sma_5, sma_60, price_to_sma_60, volatility_20..."
1,5,0.001,5,"[sma_5, sma_60, price_to_sma_60, volatility_20..."
2,5,0.005,5,"[sma_5, sma_60, price_to_sma_60, volatility_20..."
3,10,0.000,10,"[sma_5, sma_20, sma_60, price_to_sma_60, macd,..."
4,10,0.001,10,"[sma_5, sma_20, sma_60, price_to_sma_60, macd,..."
5,10,0.005,10,"[sma_5, sma_20, sma_60, price_to_sma_60, macd,..."
6,15,0.000,15,"[return_10d, return_20d, sma_5, sma_20, sma_60..."
7,15,0.001,15,"[return_10d, return_20d, sma_5, sma_20, sma_60..."
8,15,0.005,15,"[return_10d, return_20d, sma_5, sma_20, sma_60..."
9,20,0.000,20,"[return_5d, return_10d, return_20d, high_low_r..."


In [12]:
pd.set_option("display.max_colwidth", None)

display(
    sfs_xgb_results[
        ["n_features_to_select", "tol", "selected_features"]
    ]
)

,n_features_to_select,tol,selected_features
0,5,0.000,"[sma_5, sma_60, price_to_sma_60, volatility_20, volume_sma_20]"
1,5,0.001,"[sma_5, sma_60, price_to_sma_60, volatility_20, volume_sma_20]"
2,5,0.005,"[sma_5, sma_60, price_to_sma_60, volatility_20, volume_sma_20]"
3,10,0.000,"[sma_5, sma_20, sma_60, price_to_sma_60, macd, macd_signal, macd_hist, volatility_20, atr_14, volume_sma_20]"
4,10,0.001,"[sma_5, sma_20, sma_60, price_to_sma_60, macd, macd_signal, macd_hist, volatility_20, atr_14, volume_sma_20]"
5,10,0.005,"[sma_5, sma_20, sma_60, price_to_sma_60, macd, macd_signal, macd_hist, volatility_20, atr_14, volume_sma_20]"
6,15,0.000,"[return_10d, return_20d, sma_5, sma_20, sma_60, price_to_sma_20, price_to_sma_60, roc_10, roc_20, macd, macd_signal, macd_hist, volatility_20, atr_14, volume_sma_20]"
7,15,0.001,"[return_10d, return_20d, sma_5, sma_20, sma_60, price_to_sma_20, price_to_sma_60, roc_10, roc_20, macd, macd_signal, macd_hist, volatility_20, atr_14, volume_sma_20]"
8,15,0.005,"[return_10d, return_20d, sma_5, sma_20, sma_60, price_to_sma_20, price_to_sma_60, roc_10, roc_20, macd, macd_signal, macd_hist, volatility_20, atr_14, volume_sma_20]"
9,20,0.000,"[return_5d, return_10d, return_20d, high_low_range, sma_5, sma_20, sma_60, price_to_sma_5, price_to_sma_20, price_to_sma_60, rsi_14, roc_10, roc_20, macd, macd_signal, macd_hist, volatility_5, volatility_20, atr_14, volume_sma_20]"


In [13]:
sfs_xgb_results.to_csv(
    "../../data/processed/sfs_xgboost_results.csv",
    index=False
)

In [14]:
import os

path = "../../data/processed/sfs_xgboost_results.csv"

print("저장 경로:", path)
print("파일 존재 여부:", os.path.exists(path))

저장 경로: ../../data/processed/sfs_xgboost_results.csv
파일 존재 여부: True


In [15]:
for n in [5, 10, 15, 20]:
    subset = sfs_xgb_results[
        sfs_xgb_results["n_features_to_select"] == n
    ]

    print(f"\n===== {n} features =====")

    for _, row in subset.iterrows():
        print(f"tol={row['tol']}:")
        print(row["selected_features"])


===== 5 features =====
tol=0.0:
['sma_5', 'sma_60', 'price_to_sma_60', 'volatility_20', 'volume_sma_20']
tol=0.001:
['sma_5', 'sma_60', 'price_to_sma_60', 'volatility_20', 'volume_sma_20']
tol=0.005:
['sma_5', 'sma_60', 'price_to_sma_60', 'volatility_20', 'volume_sma_20']

===== 10 features =====
tol=0.0:
['sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
tol=0.001:
['sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
tol=0.005:
['sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']

===== 15 features =====
tol=0.0:
['return_10d', 'return_20d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_20', 'price_to_sma_60', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
tol=0.001:
['return_10d', 'return_20d', 'sma_5', 'sma